In [1]:
%pip install -q "monai[nibabel,tqdm,einops,gdown]"

import os, shutil, subprocess, torch, monai, nibabel, kagglehub

print("monai %s | nibabel %s | torch %s" % (monai.__version__, nibabel.__version__, torch.__version__))
print("-" * 60)

smi = shutil.which("nvidia-smi")
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available() or smi is None:
    raise SystemExit(">>> NO GPU. Set Accelerator to GPU T4 x2 in the sidebar. <<<")
print("device :", torch.cuda.get_device_name(0))
print("VRAM   : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
x = torch.randn(2000, 2000, device="cuda")
print("gpu kernels ok:", bool((x @ x).sum().isfinite()))
del x; torch.cuda.empty_cache()
print("-" * 60)

path = kagglehub.dataset_download("lnguynquangbnh/task07-pancreas")
BASE = os.path.join(path, "Task07_Pancreas")
print("dataset root:", BASE)
print("exists:", os.path.isdir(BASE))
print("/tmp free: %.0f GB" % (shutil.disk_usage("/tmp").free / 1e9))
print("-" * 60)
print("SETUP OK")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 5.6 MB/s eta 0:00:0000:0100:010m
Note: you may need to restart the kernel to use updated packages.
monai 1.6.0 | nibabel 5.4.2 | torch 2.10.0+cu128
------------------------------------------------------------
cuda available: True
device : Tesla T4
VRAM   : 15.6 GB
gpu kernels ok: True
------------------------------------------------------------
dataset root: /kaggle/input/datasets/lnguynquangbnh/task07-pancreas/Task07_Pancreas
exists: True
/tmp free: 1100 GB
------------------------------------------------------------
SETUP OK


In [2]:
import os, glob, random
import numpy as np
import nibabel as nib

imgs = sorted(f for f in glob.glob(os.path.join(BASE, "imagesTr", "*.nii"))
              if not os.path.basename(f).startswith("."))
labs = sorted(f for f in glob.glob(os.path.join(BASE, "labelsTr", "*.nii"))
              if not os.path.basename(f).startswith("."))

print("images %d | labels %d" % (len(imgs), len(labs)))
stem = lambda f: os.path.basename(f).replace(".nii.gz", "").replace(".nii", "")
print("paired :", len({stem(f) for f in imgs} & {stem(f) for f in labs}))

arr = np.asarray(nib.load(imgs[0]).dataobj)
print("sample shape %s | HU range %.0f to %.0f" % (arr.shape, arr.min(), arr.max()))

random.seed(0)
seen = set()
for f in random.sample(labs, 10):
    seen.update(np.unique(np.asarray(nib.load(f).dataobj)).astype(int).tolist())
print("label values:", sorted(seen))
print("-" * 60)
print("VERDICT:", "DATA OK" if (len(imgs) == 281 and len(labs) == 281
                                and 2 in seen and arr.min() < -500) else ">>> STOP <<<")

images 281 | labels 281
paired : 281
sample shape (512, 512, 110) | HU range -1024 to 3071
label values: [0, 1, 2]
------------------------------------------------------------
VERDICT: DATA OK


In [3]:
import os, glob, json, time, shutil
import numpy as np
import torch
from monai.utils import set_determinism
from monai.data import PersistentDataset, DataLoader, list_data_collate
from monai.networks.nets import UNet
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd, Spacingd,
    ScaleIntensityRanged, CropForegroundd, SpatialPadd,
    RandCropByLabelClassesd, RandFlipd, RandShiftIntensityd, AsDiscrete,
)

# ----------------------------- config -----------------------------
BASE      = "/kaggle/input/datasets/lnguynquangbnh/task07-pancreas/Task07_Pancreas"
OUT       = "/kaggle/working"
CACHE_TR  = "/tmp/msd_cache"
CACHE_VAL = "/tmp/msd_cache_val"
PATCH     = (96, 96, 96)
EPOCHS    = 30
VAL_EVERY = 5
LR        = 3e-4
SEED      = 0

set_determinism(seed=SEED)
dev = torch.device("cuda:0")
assert torch.cuda.is_available(), "No GPU. Stop and fix the accelerator."
print("device:", torch.cuda.get_device_name(0))

# ----------------------------- data -----------------------------
imgs = sorted(f for f in glob.glob(os.path.join(BASE, "imagesTr", "*.nii"))
              if not os.path.basename(f).startswith("."))
labs = sorted(f for f in glob.glob(os.path.join(BASE, "labelsTr", "*.nii"))
              if not os.path.basename(f).startswith("."))
data = [{"image": i, "label": l} for i, l in zip(imgs, labs)]
train_files, val_files = data[:250], data[250:]
print("train %d  val %d" % (len(train_files), len(val_files)))

train_tf = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(keys=["image", "label"], pixdim=(1.5, 1.5, 2.0), mode=("bilinear", "nearest")),
    ScaleIntensityRanged(keys=["image"], a_min=-96, a_max=215, b_min=0.0, b_max=1.0, clip=True),
    CropForegroundd(keys=["image", "label"], source_key="image", allow_smaller=True),
    SpatialPadd(keys=["image", "label"], spatial_size=PATCH),
    RandCropByLabelClassesd(keys=["image", "label"], label_key="label",
                            spatial_size=PATCH, ratios=[1, 1, 3],
                            num_classes=3, num_samples=2),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    RandShiftIntensityd(keys=["image"], offsets=0.10, prob=0.5),
])

val_tf = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(keys=["image", "label"], pixdim=(1.5, 1.5, 2.0), mode=("bilinear", "nearest")),
    ScaleIntensityRanged(keys=["image"], a_min=-96, a_max=215, b_min=0.0, b_max=1.0, clip=True),
    CropForegroundd(keys=["image", "label"], source_key="image", allow_smaller=True),
])

train_ds = PersistentDataset(train_files, transform=train_tf, cache_dir=CACHE_TR)
val_ds   = PersistentDataset(val_files,   transform=val_tf,   cache_dir=CACHE_VAL)
train_dl = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=2,
                      collate_fn=list_data_collate, pin_memory=True, drop_last=True)
val_dl   = DataLoader(val_ds, batch_size=1, num_workers=2, collate_fn=list_data_collate)

# ----------------------------- model -----------------------------
net = UNet(spatial_dims=3, in_channels=1, out_channels=3,
           channels=(16, 32, 64, 128, 256), strides=(2, 2, 2, 2),
           num_res_units=2).to(dev)
loss_fn = DiceCELoss(to_onehot_y=True, softmax=True)
opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=1e-5)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = torch.amp.GradScaler("cuda")

post_pred  = AsDiscrete(argmax=True, to_onehot=3)
post_label = AsDiscrete(to_onehot=3)
dice = DiceMetric(include_background=False, reduction="none")

def validate():
    net.eval()
    dice.reset()
    with torch.no_grad():
        for batch in val_dl:
            x = batch["image"].to(dev)
            y = batch["label"].to(dev).long()
            with torch.amp.autocast("cuda"):
                logits = sliding_window_inference(x, PATCH, 4, net, overlap=0.25)
            dice(y_pred=[post_pred(t) for t in logits.float()],
                 y=[post_label(t) for t in y])
    raw = dice.get_buffer()                       # [N, 2] with NaN where a class is absent
    m = torch.nanmean(raw.float(), dim=0)
    counted = (~torch.isnan(raw)).sum(dim=0)
    net.train()
    return float(m[0]), float(m[1]), int(counted[0]), int(counted[1])

# ----------------------------- train -----------------------------
print("-" * 70)
print("Epoch 1 includes the one-time cache build and will take roughly 8 minutes.")
print("-" * 70)

history = {"epoch": [], "loss": [], "val_epoch": [], "pancreas": [], "tumor": []}
best = -1.0
run_start = time.time()
net.train()

for ep in range(1, EPOCHS + 1):
    t0, tot, n = time.time(), 0.0, 0
    for batch in train_dl:
        x = batch["image"].to(dev, non_blocking=True)
        y = batch["label"].to(dev, non_blocking=True).long()
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda"):
            loss = loss_fn(net(x), y)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        tot += loss.detach().item()
        n += 1
    sched.step()
    avg = tot / max(n, 1)
    history["epoch"].append(ep)
    history["loss"].append(avg)
    print("epoch %3d/%d  loss %.4f  lr %.2e  %.1f s" % (
        ep, EPOCHS, avg, sched.get_last_lr()[0], time.time() - t0))

    if ep % VAL_EVERY == 0 or ep == EPOCHS:
        vt = time.time()
        p, t, np_, nt = validate()
        history["val_epoch"].append(ep)
        history["pancreas"].append(p)
        history["tumor"].append(t)
        print("   VAL  pancreas %.4f (%d vols)   tumor %.4f (%d vols)   %.1f s" % (
            p, np_, t, nt, time.time() - vt))
        score = float(np.nanmean([p, t]))
        if score > best:
            best = score
            torch.save({"epoch": ep, "state_dict": net.state_dict(),
                        "pancreas_dice": p, "tumor_dice": t},
                       os.path.join(OUT, "best_model.pth"))
            print("   saved new best (mean Dice %.4f)" % best)

torch.save({"epoch": EPOCHS, "state_dict": net.state_dict()},
           os.path.join(OUT, "last_model.pth"))
with open(os.path.join(OUT, "history.json"), "w") as f:
    json.dump(history, f, indent=2)

print("-" * 70)
print("total run time: %.1f min" % ((time.time() - run_start) / 60))
print("best mean Dice: %.4f" % best)
print("final pancreas Dice: %.4f   final tumor Dice: %.4f" % (
    history["pancreas"][-1], history["tumor"][-1]))
print("checkpoints in /kaggle/working: best_model.pth, last_model.pth, history.json")

device: Tesla T4
train 250  val 31
----------------------------------------------------------------------
Epoch 1 includes the one-time cache build and will take roughly 8 minutes.
----------------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/monai/utils/deprecate_utils.py:320: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


epoch   1/30  loss 1.3629  lr 2.99e-04  477.9 s
epoch   2/30  loss 0.8234  lr 2.97e-04  62.5 s
epoch   3/30  loss 0.6975  lr 2.93e-04  52.2 s
epoch   4/30  loss 0.6392  lr 2.87e-04  33.8 s
epoch   5/30  loss 0.6036  lr 2.80e-04  25.7 s
   VAL  pancreas 0.3268 (31 vols)   tumor 0.0000 (31 vols)   83.5 s
   saved new best (mean Dice 0.1634)
epoch   6/30  loss 0.5787  lr 2.71e-04  33.9 s
epoch   7/30  loss 0.5584  lr 2.61e-04  26.4 s
epoch   8/30  loss 0.5441  lr 2.50e-04  25.9 s
epoch   9/30  loss 0.5235  lr 2.38e-04  24.7 s
epoch  10/30  loss 0.5108  lr 2.25e-04  25.4 s
   VAL  pancreas 0.5197 (31 vols)   tumor 0.0111 (31 vols)   12.4 s
   saved new best (mean Dice 0.2654)
epoch  11/30  loss 0.4937  lr 2.11e-04  27.5 s
epoch  12/30  loss 0.4772  lr 1.96e-04  25.5 s
epoch  13/30  loss 0.4532  lr 1.81e-04  25.7 s
epoch  14/30  loss 0.4416  lr 1.66e-04  25.9 s
epoch  15/30  loss 0.4241  lr 1.50e-04  25.4 s
   VAL  pancreas 0.5019 (31 vols)   tumor 0.0140 (31 vols)   11.6 s
epoch  16/30  lo